In [ ]:
# Install needed libraries 
!pip install -q unsloth
!pip install -q opik

import json
import re
import warnings
from pathlib import Path

import opik
from opik import track
from transformers import logging as tf_logging

# Then your code
warnings.filterwarnings("ignore", category=FutureWarning)
tf_logging.set_verbosity_error()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 27.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.1 MB/s eta 

In [6]:
# Load model & tokenizer
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    device_map="auto"
)

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
# Prompt
SYSTEM_PROMPT = """
You are an assistant that outputs **only** JSON at all times. Never output any explanation, preamble, or natural language text outside the JSON.

Generate exactly two question‑answer pairs from the abstract below. The questions must require **synthesis or reasoning** – they should ask about:
- Why a certain approach was chosen (e.g., "Why does the framework use skill-level memory instead of global memory?")
- How components interact (e.g., "How do unit tests and runtime feedback enable skill refinement?")
- Comparisons with implied alternatives (e.g., "What is the advantage of a unified lifecycle over isolated skill creation?")
- Limitations or trade‑offs (e.g., "What potential limitation does the abstract acknowledge about the framework?")
- The broader implication (e.g., "How does the authors' approach advance the field of lifelong learning?")

Do NOT ask for simple definitions or factual extraction (e.g., "What is X?"). Do NOT copy whole sentences. Each answer must combine ideas from at least two different parts of the abstract, paraphrased in your own words. If the abstract does not support two high‑quality synthetic pairs, output an empty array: [].

Example of a good synthetic Q&A (for a different abstract about a new transformer variant):

{
  "question": "Why does the proposed sparse attention mechanism reduce memory usage compared to full attention?",
  "answer": "The sparse mechanism only computes attention over a fixed window and a set of global tokens, whereas full attention pairs every token with every other token. This reduces the number of pairwise computations from O(n^2) to O(n), which lowers memory footprint."
}

Now generate two such pairs for the abstract below. Output the JSON as a single line (no newlines or indentation). Do not escape single quotes (use double quotes for strings, and write apostrophes normally). Use only valid JSON syntax and Output a valid JSON array with exactly 4 objects, in this order:

[
  {"role": "user", "content": "first synthetic question"},
  {"role": "assistant", "content": "first synthetic answer (combining multiple ideas)"},
  {"role": "user", "content": "second synthetic question (different topic)"},
  {"role": "assistant", "content": "second synthetic answer"}
]

Now generate from this abstract:
"""

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "COMET_API_KEY"
secret_value = UserSecretsClient().get_secret(secret_label)

In [ ]:
# generation function 

opik.configure(
    api_key=secret_value,
    use_local=False   ,
    workspace="vaadewoyin",        # Comet workspace name
    automatic_approvals=True   # use Comet's cloud (free)
)

@track(project_name="kaggle-qa-generation")
def generate_qa(abstract):
    prompt = f"{SYSTEM_PROMPT}\n\nAbstract: {abstract}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
                **inputs,
                max_new_tokens=768, # Why: 768 tokens is sufficiently okay for generated outputs and keeps model's max seq len within 2048 limit
                do_sample=True,
                temperature=0.6,   # Why: high temperature values allows for more random response
                repetition_penalty=1.2) # Why: 1.2 is okay to discourage repetition of tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response_text

In [ ]:
def load_jsonl(file_path) -> list:
    """Read a JSONL file and return a list of records."""
    records = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                records.append(json.loads(line))
    return records

In [ ]:

def process_generated_response(response):
    """
    Extract JSON array from model response, split into two Q&A pairs,
    and return each in ChatML format: {"messages": [user, assistant]}.
    On error, returns (None, None).
    """
    # Find the first '[' and its matching ']'
    # Why: llm generates some text before & after the json array, bracket matching to extract just the needed json array
    start = response.find('[')
    if start == -1:
        return None, None
    
    bracket_count = 0
    end = start
    for i, ch in enumerate(response[start:], start=start):
        if ch == '[':
            bracket_count += 1
        elif ch == ']':
            bracket_count -= 1
            if bracket_count == 0:
                end = i
                break
    if bracket_count != 0:
        return None, None
    
    json_str = response[start:end+1]
    
    # Attempt to parse JSON
    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        # Clean common model errors and retry
        cleaned = json_str.replace("\\'", "'")
        cleaned = re.sub(r',\s*([\]}])', r'\1', cleaned)
        try:
            data = json.loads(cleaned)
        except json.JSONDecodeError:
            return None, None
    
    # Validate we have a list of 4 objects
    if not isinstance(data, list) or len(data) != 4:
        return None, None
    
    # Split into two ChatML messages
    first_pair = {"messages": data[:2]}
    second_pair = {"messages": data[2:]}
    
    return first_pair, second_pair

In [ ]:
# Paths
CLEANED_PAPERS_PATH = Path('/kaggle/input/datasets/victoradewoyin/cleaned-papers-jsonl/cleaned_papers.jsonl')
OUTPUT = Path("/kaggle/working")
QA_PAIRS_PATH = OUTPUT/"pairs.jsonl"

cleaned_papers = load_jsonl(CLEANED_PAPERS_PATH)

# Why: To resume qa generation from papers already used.
# Why //2 : each paper produces 2 pairs, so divide pair count by 2 to get paper index.
existing_count = 0
if QA_PAIRS_PATH.exists():
    existing_count = sum(1 for _ in open(QA_PAIRS_PATH))
cleaned_papers_to_process = cleaned_papers[existing_count // 2:]
    
# Why: generate qa pairs per paper & write to jsonl immediately for faster execution

for i, paper in enumerate(cleaned_papers[:1000]):
    paper_abstract = paper["abstract"]
    response = generate_qa(abstract=paper_abstract)
    first, second = process_generated_response(response)
    if first is None or second is None:
        continue
    pairs = [first, second]
    with open(QA_PAIRS_PATH, "a") as f:
         for item in pairs:
             f.write(json.dumps(item) + "\n")